# Chapter 15 &mdash; Undecidable Problems Are "$A_{TM}$ in Disguise"

**Concept 9 of the Chapter 15 decomposition:** *Undecidable Problems Are "$A_{TM}$ in Disguise"*

Every undecidability proof in the chapter traces back to $A_{TM}$ through a chain of reductions.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter15-PCP/Concept-A-TM-In-Disguise/Concept-A-TM-In-Disguise.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Step back and the chapter has one theorem and many corollaries.

$A_{TM}$ is undecidable by diagonalization (Concept 5). Everything else is reached by
a **chain of mapping reductions**:

```
A_TM  ->  Halt_TM
      ->  complement(E_TM)
      ->  PCP  ->  CFG ambiguity
                -> CFG intersection emptiness
                -> predicate-logic validity
                -> tiling the plane
```

So when you meet a new problem and suspect it is undecidable, the question is not
"how do I diagonalize?" but **"which of these is it in disguise?"**

**Rice's theorem** is the general statement for the first branch: *every*
non-trivial property of the **language** of a TM is undecidable. Most of the standard
examples are instances of it.

## 2. Definitions

### The reduction graph

In [ ]:
EDGES = [("A_TM", "Halt_TM",                  "M' halts iff M accepts"),
         ("A_TM", "complement(E_TM)",         "M_w ignores input, runs M on w"),
         ("A_TM", "PCP",                      "computation-history tiles"),
         ("PCP",  "CFG ambiguity",            "the top/bottom gadget"),
         ("PCP",  "CFG intersection empty",   "one grammar per row"),
         ("PCP",  "predicate-logic validity", "dominoes as axioms"),
         ("PCP",  "tiling the plane",         "Wang tiles")]

def reachable(edges, src):
    out, frontier = {src}, {src}
    while frontier:
        nxt = {b for a, b, _ in edges if a in frontier} - out
        out |= nxt; frontier = nxt
    return out

### Rice's theorem, as a predicate over properties

In [ ]:
def is_nontrivial_language_property(prop, examples):
    # prop maps a LANGUAGE (a set of strings) to True/False
    vals = {prop(L) for L in examples}
    return len(vals) > 1

<!-- nav-strip -->

---

&larr;&nbsp;[Ch15&nbsp;8.&nbsp;Worked Mapping Reductions: $A_{TM}\leq_m Halt_{TM}$ and $A_{TM}\leq_m \overline{E_{TM}}$](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter15-PCP/Concept-Worked-Mapping-Reductions/Concept-Worked-Mapping-Reductions.ipynb) &nbsp;&middot;&nbsp; [**Chapter 15** index](https://github.com/ganeshutah/Jove/blob/master/Chapter15-PCP/README.md) &nbsp;&middot;&nbsp; [Ch16&nbsp;1.&nbsp;Hard to Solve, Easy to Check: TSP and the Idea of a Certificate](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16-NPC/Concept-Hard-To-Solve-Easy-To-Check/Concept-Hard-To-Solve-Easy-To-Check.ipynb)&nbsp;&rarr;

---

## 3. Tests

Everything is reachable from $A_{TM}$.

In [ ]:
r = reachable(EDGES, "A_TM")
print("reachable from A_TM :")
for x in sorted(r): print("   ", x)
assert len(r) == 8

The chains, spelled out.

In [ ]:
def paths(edges, src, dst, acc=()):
    if src == dst: return [acc + (src,)]
    out = []
    for a, b, why in edges:
        if a == src and b not in acc:
            out += paths(edges, b, dst, acc + (src,))
    return out
for target in ["CFG ambiguity", "tiling the plane", "Halt_TM"]:
    for p in paths(EDGES, "A_TM", target):
        print("  " + "  ->  ".join(p))

Each edge is a computable $f$, and the edges compose.

In [ ]:
for a, b, why in EDGES:
    print("  %-22s -> %-26s  %s" % (a, b, why))
print("\n<=m is transitive, so a chain of edges IS a reduction.")

**Rice's theorem** covers the whole first branch at once.

In [ ]:
LANGS = [set(), {'0'}, {'0', '1'}, {'0' * k for k in range(5)}]
PROPS = {
 'is empty'        : lambda L: L == set(),
 'contains 0'      : lambda L: '0' in L,
 'is finite'       : lambda L: len(L) < 3,
 'TRIVIAL: True'   : lambda L: True,
 'TRIVIAL: False'  : lambda L: False,
}
for name, p in PROPS.items():
    nt = is_nontrivial_language_property(p, LANGS)
    print("  %-18s non-trivial? %-6s -> %s"
          % (name, nt, "UNDECIDABLE by Rice" if nt else "decidable (trivially)"))
assert is_nontrivial_language_property(PROPS['is empty'], LANGS)
assert not is_nontrivial_language_property(PROPS['TRIVIAL: True'], LANGS)

What Rice does **not** cover.

In [ ]:
NOT_RICE = [("does M have 7 states",       "a property of the MACHINE, not the language"),
            ("does M ever move left",      "a property of the machine's behaviour, not L(M)"),
            ("is this PCP instance solvable", "not about a machine at all")]
for a, b in NOT_RICE: print("  %-32s %s" % (a, b))
print("\nRice applies to properties of L(M).  The others need their own proofs --")
print("which is exactly why PCP is a useful separate source.")

The practical advice.

In [ ]:
print("When you meet a problem you suspect is undecidable:")
print("  1. is it a non-trivial property of L(M)?   -> Rice, done")
print("  2. is it a matching/combinatorial problem? -> try PCP")
print("  3. otherwise                               -> reduce from A_TM or Halt_TM")
print()
print("Diagonalize once.  Reduce forever.")

## 4. Exercises


1. State Rice's theorem precisely. Which two languages does the proof need?
2. Is "does $M$ halt in fewer than 100 steps?" decidable? Why does Rice not apply?
3. Add one more node to the reduction graph and justify the edge.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 254 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter15-PCP/Concept-A-TM-In-Disguise')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')